In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="07-event-recommendation",
    notebook_name="04_interview_walkthrough.ipynb"
)

# Event Recommendation System: Complete Mock Interview Walkthrough

## How to Use This Notebook

This is a **full 45-minute mock interview** for designing an event recommendation system like Eventbrite. Each section maps to how you would actually walk through this in an interview.

**Format:**
- **Interviewer** asks questions (in bold)
- **Candidate** gives answers (that is you -- practice saying these out loud)
- **Coach notes** provide tips and alternatives

---

## Time Budget (45 minutes)

| Time | Section | What to Cover |
|------|---------|---------------|
| 0-5 min | Clarifying Requirements | Ask smart questions, scope the problem |
| 5-10 min | Problem Framing | ML objective, I/O, approach (LTR) |
| 10-20 min | Feature Engineering | The star -- 5 feature categories |
| 20-30 min | Model Selection | LR -> Trees -> GBDT -> NN |
| 30-35 min | Evaluation | mAP offline, conversion rate online |
| 35-42 min | Serving Architecture | Filtering -> Ranking, online learning |
| 42-45 min | Advanced Topics | Bias, diversity, privacy, cold start |

## Phase 1: Clarifying Requirements (5 minutes)

---

**Interviewer: Design an event recommendation system for a platform like Eventbrite.**

---

In [ ]:
# Your response should cover these questions.
# Practice saying them out loud before looking at the answers.

clarifying_questions = [
    {
        "question": "What is the primary business objective?",
        "answer": "Increase ticket sales / maximize event registrations.",
        "why_ask": "Defines the ML objective. Could also be 'increase user retention' or 'maximize platform engagement' -- different objectives lead to different designs."
    },
    {
        "question": "What is the scope? Just events, or also hotels/restaurants?",
        "answer": "Just events.",
        "why_ask": "Prevents scope creep. Shows you think about boundaries."
    },
    {
        "question": "Events are ephemeral -- they happen once and expire, correct?",
        "answer": "Yes. This is a KEY insight that differentiates from movie/product recs.",
        "why_ask": "CRITICAL. This drives the entire design. Shows deep understanding."
    },
    {
        "question": "What event attributes are available?",
        "answer": "Description, price, location, date/time, category/subcategory.",
        "why_ask": "Needed for feature engineering. More attributes = richer features."
    },
    {
        "question": "Do we have labeled training data?",
        "answer": "No hand-labeled data. Use interaction logs (impressions, registrations).",
        "why_ask": "Determines how we construct training data."
    },
    {
        "question": "Do we have user location? Can users form friendships and invite others?",
        "answer": "Yes to both. Bidirectional friendships. Users can invite others.",
        "why_ask": "Location enables geo features. Social graph enables social features."
    },
    {
        "question": "What is the scale?",
        "answer": "~1M events/month, ~1M DAU.",
        "why_ask": "Affects architecture decisions (filtering needed to reduce candidate set)."
    },
    {
        "question": "Can we use external APIs like Google Maps?",
        "answer": "Yes, for distance and travel time computation.",
        "why_ask": "Enables rich location features without building our own geo system."
    }
]

print("=== Phase 1: Clarifying Questions ===")
print("(Practice asking these out loud)\n")
for i, q in enumerate(clarifying_questions, 1):
    print(f"{i}. Q: {q['question']}")
    print(f"   A: {q['answer']}")
    print(f"   Why: {q['why_ask']}")
    print()

In [ ]:
# After clarifying questions, SUMMARIZE the problem.
# This shows the interviewer you understood everything.

print("=== Problem Summary (Say This Out Loud) ===")
print()
print('"Let me summarize what we are building:')
print()
print('We need to design a personalized event recommendation system where:')
print('- Input: a user (with their profile, location, history)')
print('- Output: top-k events ranked by predicted relevance')
print('- Goal: maximize event registrations (which drives ticket sales)')
print()
print('Key constraints and characteristics:')
print('1. Events are EPHEMERAL -- they expire after they happen')
print('2. This means CONSTANT cold start for new events')
print('3. Location is critical -- events must be physically reachable')
print('4. Strong social signals -- friendships and invitations matter')
print('5. Scale: ~1M events/month, ~1M DAU')
print('6. No hand-labeled data; construct training data from interaction logs"')
print()
print("--- COACH TIP ---")
print("The summary takes 30 seconds but shows HUGE maturity.")
print("It also gives the interviewer a chance to correct any misunderstanding.")

## Phase 2: Problem Framing (5 minutes)

---

**Interviewer: Great summary. How would you frame this as an ML problem?**

---

In [ ]:
print("=== Phase 2: ML Problem Framing ===")
print()
print('"I would frame this as a Learning-to-Rank problem using a pointwise approach.')
print()
print('There are three ways to solve ranking problems:')
print()
print('1. POINTWISE: Predict a relevance score for each (user, event) pair independently.')
print('   - Simple: just binary classification (will user register? yes/no)')
print('   - Can use any classifier (LR, GBDT, NN)')
print()
print('2. PAIRWISE: Given two events, predict which the user prefers.')
print('   - More accurate rankings (RankNet, LambdaMART)')
print('   - But O(n^2) pairs and harder to train')
print()
print('3. LISTWISE: Predict the optimal ordering of an entire list.')
print('   - Directly optimizes ranking metrics (SoftRank, ListNet)')
print('   - Most complex to implement')
print()
print('I recommend starting with POINTWISE because:')
print('- Simple to implement and iterate quickly')
print('- The real value is in feature engineering (not the ranking approach)')
print('- Can upgrade to pairwise/listwise later if needed')
print()
print('Specifically: binary classification where for each (user, event) pair,')
print('the model predicts P(user registers for event). Final ranking =')
print('sort events by predicted probability, descending."')
print()
print("--- COACH TIP ---")
print("Mentioning all three approaches shows breadth of knowledge.")
print("Choosing the simplest with a clear justification shows pragmatism.")

## Phase 3: Feature Engineering (10 minutes -- the longest section!)

---

**Interviewer: What features would you use?**

---

This is the STAR of the interview. Spend the most time here.

In [ ]:
print("=== Phase 3: Feature Engineering ===")
print()
print('"Feature engineering is especially critical for event recommendation')
print('because events are ephemeral with limited interaction history.')
print('I would organize features into five categories:\n')

categories = {
    "1. LOCATION Features": [
        ("Walk score (bucketized 0-5)", "How walkable is the venue? From external API."),
        ("Walk score SIMILARITY", "Diff from user's avg walk score of past events."),
        ("Same city? (binary)", "Is the event in the user's city?"),
        ("Same country? (binary)", "Is the event in the user's country?"),
        ("Distance bucket (0-5)", "<1mi, 1-5mi, 5-20mi, 20-50mi, 50-100mi, 100+mi"),
        ("Distance SIMILARITY", "Diff from user's avg distance to past events."),
        ("Transit score, bike score", "+ their similarities (same pattern)"),
    ],
    "2. TIME Features": [
        ("Remaining time bucket (0-8)", "<1hr to 7+ days. One-hot encoded."),
        ("Remaining time SIMILARITY", "Diff from user's typical planning horizon."),
        ("Travel time bucket", "From external API (Google Maps)."),
        ("Travel time SIMILARITY", "Compared to user's historical avg."),
        ("Day-of-week profile match", "User's rate of attendance on this day."),
        ("Hour-of-day profile match", "User's rate of attendance at this hour."),
    ],
    "3. SOCIAL Features": [
        ("Num registered users", "Total people signed up for event."),
        ("Registration ratio", "Registered / impressions (conversion signal)."),
        ("Num FRIENDS registered", "How many of user's friends are going."),
        ("Friend registration ratio", "Registered friends / total friends."),
        ("Host is friend? (binary)", "Does user know the event creator?"),
        ("Past events by this host", "How often user attended host's events."),
        ("Invitations from friends", "Number of friends who invited user."),
    ],
    "4. USER Features": [
        ("Age bucket (one-hot)", "Bucketized into categories."),
        ("Gender (one-hot)", "Careful about bias with this feature!"),
    ],
    "5. EVENT Features": [
        ("Price bucket (0-4)", "Free, $1-99, $100-499, $500-1999, $2000+"),
        ("Price SIMILARITY", "Diff from user's avg event price."),
        ("Description similarity", "TF-IDF cosine sim to user's past event descriptions."),
    ]
}

for category, features in categories.items():
    print(f"\n{category}")
    for feat, desc in features:
        print(f"  - {feat}: {desc}")

print('\n"')
print("\n--- COACH TIP ---")
print("The SIMILARITY features are the secret weapon. They compare each event")
print("attribute to the user's HISTORICAL PREFERENCE, not just the raw value.")
print("This is what makes the system truly personalized.")

In [ ]:
# Follow-up questions the interviewer might ask about features:

followups = {
    "How do you handle batch vs streaming features?": {
        "answer": (
            "Static features (age, gender, event description) are computed periodically "
            "by batch jobs and stored in a Feature Store. Dynamic features (registered "
            "count, remaining time) are computed in real-time by a streaming pipeline "
            "(e.g., Kafka + Flink)."
        )
    },
    "How do you handle feature computation efficiency?": {
        "answer": (
            "Computing distance via API for every (user, event) pair is expensive at scale. "
            "Two options: (1) Pre-compute and cache distances in batch, (2) Pass raw "
            "lat/lon to the model and let it learn the distance function. Option 1 for "
            "existing events, option 2 as fallback for brand-new events."
        )
    },
    "What about decay for user preferences?": {
        "answer": (
            "User preferences change over time. I use an exponential decay factor when "
            "computing similarity features: recent interactions get higher weight. "
            "The half-life is a tunable hyperparameter -- typically 30-90 days."
        )
    },
    "Any concerns about bias from user attributes?": {
        "answer": (
            "Yes! Using age and gender as features can create discriminatory recommendations. "
            "For example, not showing tech events to women or not showing concerts to seniors. "
            "We should monitor recommendation distributions across demographics and consider "
            "fairness-aware training or removing protected attributes."
        )
    }
}

print("=== Common Follow-Up Questions on Features ===")
for q, details in followups.items():
    print(f"\nQ: {q}")
    print(f"A: {details['answer']}")

## Phase 4: Model Selection (10 minutes)

---

**Interviewer: What model would you use?**

---

In [ ]:
print("=== Phase 4: Model Selection ===")
print()
print('"I would discuss four model families and recommend a two-phase approach:\n')

models = [
    ("Logistic Regression", "BASELINE", 
     "Fast, interpretable, but cannot learn non-linear feature interactions. "
     "Good starting point to validate the feature pipeline."),
    ("Decision Trees", "NOT RECOMMENDED alone",
     "Overfits easily, axis-parallel boundaries. But the ENSEMBLE versions are great."),
    ("GBDT (XGBoost)", "STRONG BASELINE",
     "Reduces both bias and variance. Works great with structured features. "
     "Fast to implement. XGBoost variant has proven strong in competitions. "
     "Limitation: cannot do continual learning -- must retrain from scratch."),
    ("Neural Network", "PRODUCTION MODEL",
     "Supports continual learning (critical for ephemeral events). "
     "Learns non-linear patterns. Can incorporate embeddings. "
     "Needs more data prep and compute, but we have massive interaction data.")
]

for model, verdict, explanation in models:
    print(f"  {model}: [{verdict}]")
    print(f"    {explanation}")
    print()

print('My recommendation:')
print('  Phase 1: Start with XGBoost as baseline.')
print('    - Fast to implement and train')
print('    - Works well with our structured features')
print('    - Gives us an initial baseline to beat')
print()
print('  Phase 2: Move to Neural Network for production.')
print('    - CONTINUAL LEARNING: can fine-tune on new data without retraining')
print('    - This is critical because events are ephemeral')
print('    - New events and new user behavior appear constantly')
print('    - Massive training data from user interactions justifies NN')
print()
print('Loss function: Binary Cross-Entropy (standard for binary classification).')
print()
print('Class imbalance handling: Users explore many events before registering,')
print('so negatives vastly outnumber positives. Use focal loss, class-balanced loss,')
print('or undersample the majority class."')
print()
print("--- COACH TIP ---")
print("The KEY differentiator is CONTINUAL LEARNING.")
print("GBDT cannot do it. NN can. For ephemeral events, this is crucial.")
print("Make sure to emphasize this -- it is the strongest argument for NN.")

In [ ]:
# If interviewer asks about bagging vs boosting:

print("=== Bagging vs Boosting (If Asked) ===")
print()
print('"Bagging (e.g., Random Forest):')
print('  - Trains multiple trees in PARALLEL on different data subsets')
print('  - Combines predictions by voting/averaging')
print('  - Reduces VARIANCE (overfitting)')
print('  - Does NOT help with bias (underfitting)')
print()
print('Boosting (e.g., XGBoost, AdaBoost):')
print('  - Trains weak classifiers SEQUENTIALLY')
print('  - Each one fixes the mistakes of the previous')
print('  - Reduces BOTH bias and variance')
print('  - Slower training (sequential), but more accurate')
print()
print('Why boosting is preferred in practice:')
print('  Bagging only helps with high variance.')
print('  Boosting helps with both high bias AND high variance.')
print('  For structured tabular data, GBDT/XGBoost is often the best')
print('  non-neural approach."')

## Phase 5: Evaluation (5 minutes)

---

**Interviewer: How would you evaluate this system?**

---

In [ ]:
print("=== Phase 5: Evaluation ===")
print()
print('"I would use both offline and online metrics:\n')

print('OFFLINE METRICS (for model development):')
print()
print('  Precision@k / Recall@k: NOT ideal.')
print('    They do not consider RANKING QUALITY -- just whether')
print('    relevant items appear in the top-k.')
print()
print('  MRR (Mean Reciprocal Rank): NOT ideal.')
print('    Focuses on rank of the FIRST relevant item.')
print('    In event recommendation, MULTIPLE events can be relevant.')
print()
print('  nDCG: Good, but works best with GRADED relevance (1-5 stars).')
print()
print('  >>> mAP (Mean Average Precision): BEST FIT.')
print('    Works with BINARY relevance (registered=1 or not=0).')
print('    Rewards putting relevant items HIGHER in the ranking.')
print('    Handles multiple relevant items per query.')

print()
print('ONLINE METRICS (for production):"')
print()

online_metrics = [
    ("CTR", "clicked / impressions", "Are users clicking? But beware of clickbait."),
    ("Conversion Rate", "registrations / impressions", "MOST IMPORTANT. Are users actually signing up?"),
    ("Bookmark Rate", "bookmarked / impressions", "Interest signal without commitment."),
    ("Revenue Lift", "(new_rev - old_rev) / old_rev", "Direct business impact measurement."),
]

for metric, formula, why in online_metrics:
    print(f"  {metric}: {formula}")
    print(f"    Why: {why}")
    print()

print("--- COACH TIP ---")
print("Key insight to share: Why mAP over nDCG?")
print("Because our relevance is BINARY (registered or not), not graded.")
print("Also: Why not just CTR? Clickbait events inflate CTR without real value.")

## Phase 6: Serving Architecture (7 minutes)

---

**Interviewer: How would you serve this in production?**

---

In [ ]:
print("=== Phase 6: Serving Architecture ===")
print()
print('"The system has two main pipelines:\n')

print('PIPELINE 1: ONLINE LEARNING')
print('  Because events are ephemeral and cold-start is constant,')
print('  the model must be CONTINUOUSLY fine-tuned.')
print()
print('  Flow: New interactions (clicks, registrations, new events)')
print('    -> Feature computation pipeline')
print('    -> Fine-tune neural network on new data')
print('    -> Evaluate on held-out set')
print('    -> If improved, deploy to production')
print()
print('  Key: NN supports fine-tuning without full retraining.')
print('  GBDT would require expensive full retraining.')

print()
print('PIPELINE 2: PREDICTION (when user opens the app)')
print()
print('  Step 1: EVENT FILTERING')
print('    Input: 1 million total events')
print('    Apply simple rules:')
print('      - Location filter (same city/region)')
print('      - Category filter (user preferences)')
print('      - Date filter (not expired, within reasonable future)')
print('      - User-applied filters (e.g., "concerts only")')
print('    Output: ~100-500 candidate events')
print()
print('  Step 2: FEATURE COMPUTATION')
print('    For each (user, candidate_event) pair:')
print('      - Static features: read from Feature Store (Redis/DynamoDB)')
print('      - Dynamic features: compute real-time (registered count, remaining time)')
print()
print('  Step 3: RANKING')
print('    Model predicts P(register) for each (user, event) pair')
print('    Sort by predicted probability, descending')
print('    Return top-k events to the user"')
print()
print("--- COACH TIP ---")
print("The filtering step is ESSENTIAL. Without it, you would need to")
print("score 1M (user, event) pairs per request -- too expensive.")
print("Filtering reduces the candidate set by 1000x.")

In [ ]:
# Draw the architecture for the interviewer

print("=== Architecture Diagram (Draw This on the Whiteboard) ===")
print()
print("""                        USER REQUEST
                             |
                             v
    +========================+========================+
    |                 PREDICTION PIPELINE              |
    |                                                  |
    |   +------------+    +----------+    +---------+  |
    |   | Event      |    | Feature  |    | Ranking |  |
    |   | Filtering  |--->| Compute  |--->| Model   |  |
    |   | (1M->500)  |    |          |    | (NN)    |  |
    |   +------------+    +----+-----+    +---------+  |
    |                          |                       |
    |              +-----------+-----------+            |
    |              |                       |            |
    |      +-------+------+    +-----------+-----+     |
    |      | Feature      |    | Real-time       |     |
    |      | Store        |    | Feature Compute  |    |
    |      | (static)     |    | (dynamic)        |    |
    |      +--------------+    +-----------------+     |
    +==================================================+
                             |
                             v
                        TOP-K EVENTS
    
    +==================================================+
    |              ONLINE LEARNING PIPELINE             |
    |                                                   |
    |   New Interactions --> Feature Pipeline            |
    |       --> Fine-tune NN --> Evaluate --> Deploy     |
    +===================================================+
""")

## Phase 7: Advanced Topics (3 minutes)

---

**Interviewer: What other considerations would you discuss?**

---

In [ ]:
print("=== Phase 7: Advanced Topics ===")
print()
print('"Here are additional considerations I would like to highlight:\n')

topics = [
    {
        "topic": "Bias in Recommendations",
        "key_point": (
            "Position bias (top items get more clicks), popularity bias (rich-get-richer), "
            "and demographic bias from user attributes. Mitigate with inverse propensity "
            "weighting, fairness constraints, and regular bias audits."
        )
    },
    {
        "topic": "Feature Crossing",
        "key_point": (
            "Crossing features like (age x category) or (distance x price) captures "
            "interactions that individual features miss. NNs learn this implicitly; "
            "tree models benefit from explicit crosses."
        )
    },
    {
        "topic": "Diversity and Freshness",
        "key_point": (
            "Users want variety. Use MMR (Maximal Marginal Relevance) or "
            "category-aware re-ranking to avoid showing only one type of event. "
            "Also use epsilon-greedy exploration to discover new user interests."
        )
    },
    {
        "topic": "Privacy and Security",
        "key_point": (
            "We use live location and personal attributes. Consider differential privacy, "
            "data minimization, location obfuscation, and clear user consent mechanisms."
        )
    },
    {
        "topic": "Two-Sided Marketplace",
        "key_point": (
            "Hosts are suppliers, users are demand. Optimizing only for users creates "
            "unfairness -- popular hosts dominate. Ensure minimum impression guarantees "
            "for all active hosts, especially new ones."
        )
    },
    {
        "topic": "Data Leakage",
        "key_point": (
            "When constructing training data, ensure no future information leaks. "
            "For example, the registration count feature should NOT include the target "
            "user's own registration. Use careful timestamp-based splitting."
        )
    },
    {
        "topic": "Model Update Frequency",
        "key_point": (
            "How often to update the model? With NN and continual learning, "
            "we can update multiple times per day. The key is having a robust "
            "evaluation pipeline to catch regressions before deployment."
        )
    }
]

for t in topics:
    print(f"  {t['topic']}:")
    print(f"    {t['key_point']}")
    print()

print("--- COACH TIP ---")
print("You do not need to cover ALL of these. Pick 2-3 that you are")
print("most comfortable with. The interviewer will ask follow-ups on")
print("the ones they care about most.")

## The 30-Second Elevator Pitch

If you had to summarize the ENTIRE design in 30 seconds:

In [ ]:
print("="*60)
print("  THE 30-SECOND ELEVATOR PITCH")
print("="*60)
print()
print('"We build a pointwise Learning-to-Rank system that predicts')
print('the probability a user will register for each candidate event.')
print()
print('We engineer features across five categories -- location, time,')
print('social, user, and event -- with SIMILARITY features that compare')
print('each event to the user\'s historical preferences. This handles')
print('the cold-start problem inherent in ephemeral events.')
print()
print('We start with XGBoost as a baseline, then move to a neural')
print('network for continual learning -- critical because new events')
print('appear daily.')
print()
print('The serving architecture filters 1M events down to hundreds')
print('using simple rules, then ranks them with the model.')
print()
print('We evaluate offline with mAP (binary relevance for registration)')
print('and online with conversion rate and revenue lift."')
print()
print("="*60)

## Common Curveball Questions

Here are tricky questions an interviewer might throw at you:

In [ ]:
curveballs = {
    "What if a user has NO history (new user cold start)?": {
        "answer": (
            "For new users, we fall back to: "
            "(1) Location-based recommendations (popular nearby events), "
            "(2) Demographic-based (age/gender cohort preferences), "
            "(3) Onboarding survey (ask user to pick favorite categories). "
            "As the user interacts, personalization improves rapidly."
        )
    },
    "How do you handle events that are free vs expensive?": {
        "answer": (
            "Price bucket and price similarity features handle this. "
            "A user who always attends free events has a high price similarity "
            "score for free events and low for expensive ones. "
            "The model learns this pattern automatically."
        )
    },
    "What if the model keeps recommending the same events?": {
        "answer": (
            "Three mechanisms: (1) MMR re-ranking for diversity, "
            "(2) Category cap (max N events per category), "
            "(3) De-duplication (do not show events already seen/dismissed). "
            "Also use exploration (epsilon-greedy) to discover new interests."
        )
    },
    "How do you prevent data leakage?": {
        "answer": (
            "Use strict timestamp-based splitting. Training data uses only "
            "interactions BEFORE a cutoff date; test data uses AFTER. "
            "Also ensure features like 'num_registered' do not include "
            "the target user's own registration. And historical features "
            "only use events that occurred BEFORE the candidate event."
        )
    },
    "How would you A/B test this?": {
        "answer": (
            "Split users into control (current system) and treatment (new model). "
            "Run for at least 2 weeks. Primary metric: conversion rate. "
            "Secondary: CTR, bookmark rate, revenue. "
            "Watch for novelty effects (users clicking more just because it is new). "
            "Also monitor negative metrics: unsubscribe rate, app deletion."
        )
    },
    "What about real-time events (e.g., happening in 1 hour)?": {
        "answer": (
            "The remaining_time feature handles this. Events starting soon "
            "get a different bucket (0 = <1hr). The model learns that some "
            "users are spontaneous (like last-minute events) while others "
            "are planners (prefer events 7+ days out). The similarity feature "
            "captures this personal preference."
        )
    },
    "How would you scale to 100M events?": {
        "answer": (
            "The filtering step is the key. Use geo-hashing to efficiently "
            "filter by location. Shard the event index by region. "
            "Use approximate nearest neighbors for embedding-based retrieval. "
            "The ranking model only needs to score 100-500 candidates, "
            "not 100M."
        )
    }
}

print("=== Curveball Questions & Answers ===")
for q, details in curveballs.items():
    print(f"\nQ: {q}")
    print(f"A: {details['answer']}")

## Self-Assessment Checklist

Use this after your practice run to see how you did:

In [ ]:
checklist = [
    # Requirements
    ("Requirements", "Asked about business objective"),
    ("Requirements", "Identified events as ephemeral (KEY insight)"),
    ("Requirements", "Asked about scale"),
    ("Requirements", "Asked about available data and labels"),
    ("Requirements", "Summarized the problem clearly"),
    
    # ML Framing
    ("ML Framing", "Defined ML objective (maximize registrations)"),
    ("ML Framing", "Specified input/output (user -> ranked events)"),
    ("ML Framing", "Mentioned LTR approaches (pointwise/pairwise/listwise)"),
    ("ML Framing", "Justified pointwise choice"),
    
    # Features
    ("Features", "Covered location features (distance, walk score, same city)"),
    ("Features", "Covered time features (remaining time, day/hour profiles)"),
    ("Features", "Covered social features (friends, invitations, host)"),
    ("Features", "Covered user features (age, gender)"),
    ("Features", "Covered event features (price, description similarity)"),
    ("Features", "Explained SIMILARITY features (historical preference comparison)"),
    ("Features", "Discussed batch vs streaming features"),
    
    # Model
    ("Model", "Discussed multiple model families"),
    ("Model", "Explained why GBDT is a good baseline"),
    ("Model", "Explained why NN is needed (CONTINUAL LEARNING)"),
    ("Model", "Mentioned class imbalance and solutions"),
    ("Model", "Mentioned binary cross-entropy loss"),
    
    # Evaluation
    ("Evaluation", "Explained why mAP (not nDCG, not MRR)"),
    ("Evaluation", "Listed online metrics (CTR, conversion rate, revenue)"),
    ("Evaluation", "Explained why conversion rate > CTR"),
    
    # Architecture
    ("Architecture", "Described filtering step (1M -> hundreds)"),
    ("Architecture", "Described ranking step (model scoring)"),
    ("Architecture", "Described online learning pipeline"),
    ("Architecture", "Mentioned feature store (static) vs real-time (dynamic)"),
    
    # Advanced
    ("Advanced", "Discussed at least one type of bias"),
    ("Advanced", "Mentioned diversity/exploration"),
    ("Advanced", "Mentioned cold-start strategy"),
]

print("=== Self-Assessment Checklist ===")
print("(Check off each item after your practice run)\n")
current_category = ""
for category, item in checklist:
    if category != current_category:
        print(f"\n{category}:")
        current_category = category
    print(f"  [ ] {item}")

print(f"\nTotal items: {len(checklist)}")
print(f"Target: Hit at least {int(len(checklist) * 0.8)} items ({int(len(checklist) * 0.8)}/{len(checklist)}) for a strong interview.")

## Final Tips

1. **Start with the problem, not the solution.** Clarify requirements FIRST.

2. **Feature engineering is the star.** Spend the most time here. The five categories + similarity features are the unique value.

3. **Always mention the ephemeral nature of events.** This is THE key insight that differentiates this from standard recommendation.

4. **Justify your choices.** Do not just say "I would use XGBoost." Say WHY: structured data, fast to implement, strong baseline. Then say why you would UPGRADE to NN: continual learning for ephemeral events.

5. **Draw the architecture.** A simple diagram of filtering -> ranking with the two pipelines (prediction + online learning) is worth 100 words.

6. **Know the metrics deeply.** Why mAP over nDCG? Binary relevance. Why conversion rate over CTR? Clickbait resistance.

7. **Have 2-3 advanced topics ready.** Bias, diversity, cold-start, two-sided marketplace, data leakage. Pick the ones you know best.

8. **Practice out loud.** Reading silently is not the same as speaking. Time yourself. The 45-minute structure should feel natural.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)